# heli-lto — smoke test in Google Colab

Verifies the `foca_heli` package by uploading the source zip directly into the Colab runtime, no GitHub clone needed.

## How to use

1. Run cell 1. It opens an upload dialog — pick `heli-lto.zip` from your local machine.
2. Run remaining cells top to bottom. Each prints what it found and asserts correctness.

Requires Python ≥ 3.10 (Colab default Python is fine — currently 3.11). No third-party dependencies.

## 1. Upload the package zip

Click "Choose Files" when prompted, select your local `heli-lto.zip`.

In [ ]:
from google.colab import files
import zipfile
import os
import sys
import shutil

# Clean any previous run
if os.path.exists("/content/heli-lto"):
    shutil.rmtree("/content/heli-lto")

uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print(f"Uploaded: {zip_name} ({len(uploaded[zip_name])} bytes)")

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("/content")

PACKAGE_ROOT = "/content/heli-lto"
assert os.path.exists(os.path.join(PACKAGE_ROOT, "foca_heli", "__init__.py")), (
    f"Did not find foca_heli/__init__.py inside {PACKAGE_ROOT}. "
    "The zip should extract to a heli-lto/ folder containing foca_heli/."
)

if PACKAGE_ROOT not in sys.path:
    sys.path.insert(0, PACKAGE_ROOT)

print(f"Extracted to: {PACKAGE_ROOT}")
print(f"Python version: {sys.version_info[:3]}")
assert sys.version_info >= (3, 10), "foca_heli requires Python >= 3.10"

## 2. Import and check public API

In [ ]:
import foca_heli
from foca_heli import (
    make_strategy, compute_lto, lto_to_dict,
    PROFILES, derive_category,
    build_departure, build_arrival, to_wkt_linestring_z, to_geojson_feature,
    read_engines_csv, write_lto_csv,
    lookup_airframe, lookup_mtom_kg, BUILT_IN_AIRFRAME_MTOMS,
)

print(f"Loaded foca_heli from: {foca_heli.__file__}")
print(f"Public API symbols:    {len(foca_heli.__all__)}")
print(f"Built-in MTOM lookup:  {len(BUILT_IN_AIRFRAME_MTOMS)} entries")
print(f"Categories:            {list(PROFILES.keys())}")

## 3. Run the package's own test suite

41 tests covering formula correctness, CSV round-trips, trajectory shape, reference-data sync. If anything fails, the rest of the notebook is suspect.

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "tests.test_all"],
    cwd=PACKAGE_ROOT,
    capture_output=True, text=True,
)
print("STDOUT:")
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
print("\nSTDERR (last lines):")
print("\n".join(result.stderr.split("\n")[-15:]))
assert result.returncode == 0, "Test suite failed"
print("\nTests OK")

## 4. FOCA 2015 worked examples

Reproduces the three appendix cases. CO2 sanity check applies the 3160 g/kg Jet-A factor.

In [ ]:
validation_cases = [
    # (label,    category,                  max_shp, n_eng, target_fuel_kg, app)
    ("AS350B2",  "SINGLE_TURBOSHAFT",       732,     1,     24.7,           "A"),
    ("A109",     "TWIN_TURBOSHAFT_LIGHT",   550,     2,     36.5,           "B"),
    ("AS332",    "TWIN_TURBOSHAFT_HEAVY",   1820,    2,     78.4,           "C"),
]

print(f"{'Helicopter':<10} {'App':<4} {'FOCA':>8} {'Computed':>10} {'Delta':>8} {'CO2(kg)':>9}")
print("-" * 56)
all_ok = True
for label, cat, shp, n, target, app in validation_cases:
    s = make_strategy(category=cat, max_shp_per_engine=shp, number_of_engines=n)
    r = compute_lto(s)
    delta_pct = (r.fuel_kg - target) / target * 100
    fuel_ok = abs(delta_pct) < 3.0
    expected_co2 = r.fuel_kg * 3160.0   # Jet-A factor in g/kg
    co2_ok = abs(r.co2_g - expected_co2) < 1.0
    flag = "OK" if (fuel_ok and co2_ok) else "FAIL"
    print(f"{label:<10} {app:<4} {target:>6.1f}kg {r.fuel_kg:>8.2f}kg "
          f"{delta_pct:>+6.1f}% {r.co2_g/1000:>7.1f}  {flag}")
    all_ok &= fuel_ok and co2_ok

assert all_ok, "At least one validation case failed"
print("\nAll three FOCA worked examples reproduced within tolerance.")

## 5. Per-mode breakdown

Shows the GI / TO / AP split for one helicopter.

In [ ]:
s = make_strategy(category="TWIN_TURBOSHAFT_HEAVY", max_shp_per_engine=1820, number_of_engines=2)
r = compute_lto(s)

print(f"AS332 / MAKILA 1A1 — twin turboshaft heavy, 1820 SHP × 2\n")
print(f"{'Mode':<6} {'Pwr':>5} {'Time(s)':>8} {'Fuel(kg)':>10} {'NOx(g)':>9} {'HC(g)':>9} {'CO(g)':>9} {'PM(g)':>8} {'CO2(kg)':>9}")
print("-" * 80)
for m in r.modes:
    print(f"{m.mode:<6} {int(m.power_fraction*100):>4}% {m.time_s:>8.0f} "
          f"{m.fuel_kg:>10.2f} {m.nox_g:>9.2f} {m.hc_g:>9.2f} {m.co_g:>9.2f} "
          f"{m.pm_g:>8.3f} {m.co2_g/1000:>9.2f}")
print("-" * 80)
print(f"{'TOTAL':<6} {'':>5} {'':>8} {r.fuel_kg:>10.2f} {r.nox_g:>9.2f} {r.hc_g:>9.2f} "
      f"{r.co_g:>9.2f} {r.pm_g:>8.3f} {r.co2_g/1000:>9.2f}")

## 6. CSV round-trip

Builds a small fleet CSV in OLD plugin schema, exercises the auto-detecting reader, runs LTO on each row, writes results CSV. Verifies the airframe MTOM lookup correctly classifies twin turboshafts.

In [ ]:
import tempfile

csv_text = """engine_name,engine_type,max_shp_per_engine,number_of_engines
HIO-360,PISTON,190,1
ARRIEL 1D1,TURBOSHAFT,732,1
PW206C,TURBOSHAFT,550,2
MAKILA 1A1,TURBOSHAFT,1820,2
PT6C-67C,TURBOSHAFT,1100,2
"""

tmp_in = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
tmp_in.write(csv_text)
tmp_in.close()

rows = read_engines_csv(tmp_in.name)
print(f"Read {len(rows)} engines:\n")
print(f"{'Engine':<14} {'Category':<24} {'MTOM (kg)':>10}")
print("-" * 51)
for r in rows:
    mtom_str = f"{r.mtom_kg:.0f}" if r.mtom_kg else "(N/A)"
    print(f"{r.engine_name:<14} {r.category:<24} {mtom_str:>10}")

# Compute LTO for each
results = []
for engine_row in rows:
    strategy = make_strategy(
        category=engine_row.category,
        max_shp_per_engine=engine_row.max_shp_per_engine,
        number_of_engines=engine_row.number_of_engines,
    )
    # write_lto_csv expects dict form via lto_to_dict
    results.append((engine_row, lto_to_dict(compute_lto(strategy))))

tmp_out = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
tmp_out.close()
write_lto_csv(tmp_out.name, results)

with open(tmp_out.name) as fp:
    out_text = fp.read()

# Show only the totals columns for readability
import csv as _csv
out_rows = list(_csv.DictReader(out_text.splitlines()))
print(f"\nLTO totals from output CSV:\n")
print(f"{'Engine':<14} {'Fuel(kg)':>10} {'NOx(g)':>9} {'CO2(kg)':>9}")
print("-" * 45)
for row in out_rows:
    print(f"{row['engine_name']:<14} {float(row['lto_fuel_kg']):>10.2f} "
          f"{float(row['lto_nox_g']):>9.1f} {float(row['lto_co2_g'])/1000:>9.2f}")

os.unlink(tmp_in.name)
os.unlink(tmp_out.name)

## 7. Trajectory geometry

Builds a departure trajectory for the AS332 (twin heavy) and exports as WKT. Vertices are at mode-transition points; trajectory ends at FOCA's 914 m (3000 ft) LTO ceiling.

In [ ]:
import json

departure = build_departure("TWIN_TURBOSHAFT_HEAVY")
print(f"Departure: {len(departure)} vertices\n")
print(f"{'Mode':<6} {'t(s)':>6} {'X(m)':>8} {'Y(m)':>6} {'Z(m)':>6} {'TAS(m/s)':>9}")
print("-" * 45)
for p in departure:
    print(f"{p.mode:<6} {p.t_s:>6.0f} {p.x_m:>8.0f} {p.y_m:>6.0f} {p.z_m:>6.0f} {p.tas_m_s:>9.1f}")

assert abs(departure[-1].z_m - 914.0) < 1.0, (
    f"Final altitude {departure[-1].z_m} m not at FOCA LTO ceiling (914 m)"
)

wkt = to_wkt_linestring_z(departure)
print(f"\nWKT (truncated): {wkt[:120]}...")

gj = to_geojson_feature(departure, properties={"phase": "departure", "category": "TWIN_TURBOSHAFT_HEAVY"})
print(f"\nGeoJSON has {len(gj['geometry']['coordinates'])} coordinate triples")
assert json.loads(json.dumps(gj))   # serializable round-trip

## 8. Optional — visualize the trajectory

Uses matplotlib (preinstalled in Colab) for a quick 2D side-view of departure + arrival altitude profiles.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

for label, color in [("PISTON", "C0"), ("SINGLE_TURBOSHAFT", "C1"),
                      ("TWIN_TURBOSHAFT_LIGHT", "C2"), ("TWIN_TURBOSHAFT_HEAVY", "C3")]:
    dep = build_departure(label)
    arr = build_arrival(label)
    # Plot departure as solid, arrival as dashed (mirrored to negative x for visualization)
    ax.plot([p.x_m / 1000 for p in dep], [p.z_m for p in dep],
            color=color, label=f"{label} dep", linewidth=2)
    ax.plot([-p.x_m / 1000 for p in arr], [p.z_m for p in arr],
            color=color, linestyle="--", label=f"{label} arr", linewidth=2)

ax.axhline(914, color="gray", linestyle=":", alpha=0.5, label="LTO ceiling 914 m")
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Ground distance from helipad (km) — arrival left, departure right")
ax.set_ylabel("Altitude AGL (m)")
ax.set_title("FOCA 2015 LTO trajectories by helicopter category")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary

If all cells above ran without error, the package works. The CO2 fix is verified (cell 4 includes the `r.co2_g == r.fuel_kg × 3160` assertion).

Next steps for production use:

- **In another Colab notebook**: just upload the zip and `sys.path.insert` again. No need to install.
- **Local dev**: `pip install -e .` from the extracted folder; then `from foca_heli import ...` works directly without the path hack.
- **Once the GitHub repo is public**: `pip install git+https://github.com/eurocontrol-asu/heli-lto.git`.